# Databricks Vector Search Index 개요

> **목적**: AI Agent(RAG 등)를 위한 Databricks Mosaic AI Vector Search의 아키텍처와 하이브리드 검색 메커니즘을 이해합니다.

---

## 1. Mosaic AI Vector Search란?

**Mosaic AI Vector Search**는 Databricks Data Intelligence Platform에 내장된 **Databricks의 서버리스 벡터 검색 엔진**입니다.

| 특징 | 설명 |
|---|---|
| **Unity Catalog 통합** | Delta 테이블 기반으로 인덱스 생성, 거버넌스(ACL) 자동 적용 |
| **자동 동기화** | 소스 Delta 테이블 변경 시 인덱스 자동 업데이트 |
| **서버리스** | 별도 인프라 관리 없이 엔드포인트 자동 프로비저닝 |
| **하이브리드 검색** | 유사도 검색 + 키워드 검색 동시 지원 |
| **필터링 & 리랭킹** | 메타데이터 필터 및 결과 재순위 지원 |

---
## 2. AI Agent를 위한 Vector Search 아키텍처

AI Agent(RAG)가 Vector Search Index를 활용하는 **End-to-End 아키텍처**는 다음과 같습니다.

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/databricks ai agent end to end.png" width="800" />
    </td>
  </tr>
</table>
<table>

### 핵심 컴포넌트 설명

| 컴포넌트 | 역할 | Databricks 서비스 |
|---|---|---|
| **Embedding Model** | 텍스트→벡터 변환 | Foundation Model Serving Endpoint (e.g. `databricks-gte-large-en`) |
| **Vector Search Endpoint** | 인덱스 호스팅 및 쿼리 처리 | Serverless 자동 프로비저닝 |
| **Vector Search Index** | 임베딩 + 메타데이터 저장 | Delta 테이블 기반, Unity Catalog 관리 |
| **LLM** | 컨텍스트 기반 응답 생성 | Foundation Model API / External Model |

### Databricks 환경에서의 데이터 흐름

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/databricks vector search index flow.png" width="800" />
    </td>
  </tr>
</table>
<table>

---
## 3. Vector Search Index 유형 (3가지)

Databricks는 임베딩 제공 방식에 따라 **3가지 인덱스 유형**을 지원합니다.

### 3-1. Delta Sync Index — Managed Embeddings (추천)

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Managed Embeddings.png" width="400" />
    </td>
  </tr>
</table>

* 소스 테이블에 **텍스트 컬럼만** 제공하면 Databricks가 자동으로 임베딩 계산
* 임베딩 모델 지정 필요 (e.g. `databricks-gte-large-en`)
* Delta 테이블 변경 시 **자동 동기화**
* 가장 간편한 방법 

### 3-2. Delta Sync Index — Self-managed Embeddings

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Self-managed Embeddings.png" width="400" />
    </td>
  </tr>
</table>

* 소스 테이블에 **사전 계산된 임베딩 칼럼**을 포함
* 외부 임베딩 모델(OpenAI, Cohere 등) 사용 시 유용
* Delta 테이블 변경 시 자동 동기화
* ⚠️ Self-managed → Managed로 변환 **불가** (새 인덱스 생성 필요)

### 3-3. Direct Vector Access Index

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Direct Vector Access Index.png" width="400" />
    </td>
  </tr>
</table>

* Delta 테이블 없이 **REST API로 직접** 임베딩 삽입/삭제
* 자동 동기화 없음 — 완전한 수동 관리
* 실시간 데이터 스트림 등 특수 케이스에 적합

### 비교 요약

| 구분 | Managed Embeddings | Self-managed Embeddings | Direct Vector Access |
|---|---|---|---|
| **임베딩 계산** | Databricks 자동 | 사용자 직접 | 사용자 직접 |
| **소스 테이블** | Delta Table (텍스트) | Delta Table (텍스트+임베딩) | 없음 |
| **자동 동기화** | ✅ Yes | ✅ Yes | ❌ No |
| **난이도** | ⭐ 낮음 | ⭐⭐ 중간 | ⭐⭐⭐ 높음 |
| **적합 케이스** | 빠른 프로토타입, RAG | 커스텀 임베딩 모델 | 실시간 스트림 |

---
## 4. 하이브리드 검색: 유사도 검색 + 키워드 검색

Databricks Vector Search는 **Hybrid Search**를 지원합니다.  
두 가지 검색 방식을 결합하여 단독 사용 대비 **더 높은 검색 정확도**를 달성합니다.

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/Hybrid Search.png" width="400" />
    </td>
  </tr>
</table>

---

### 4-1. 유사도 검색 (Similarity Search)

벡터 공간에서 의미적으로 가까운 문서를 찾는 방법입니다.

#### 핵심 알고리즘: ANN (Approximate Nearest Neighbor)

| 항목 | 설명 |
|---|---|
| **목적** | 고차원 벡터 공간에서 쿼리와 가장 유사한 K개 벡터를 빠르게 찾음 |
| **방법** | 정확한 KNN 대신 **근사값**을 반환하여 속도 확보 |
| **대표 기법** | HNSW (Hierarchical Navigable Small World) |
| **시간복잡도** | O(log n) — 수십억 벡터에서도 ms 단위 응답 |

#### 유사도 측정 방법 (Distance Metrics)

| 메트릭 | 수식 | 특징 | 적합 케이스 |
|---|---|---|---|
| **Cosine Similarity** | cos(θ) = (A·B) / (‖A‖·‖B‖) | 방향(의미) 비교, 크기 무시 | 텍스트 임베딩 (가장 널리 사용) |
| **Euclidean (L2)** | d = √Σ(aᵢ-bᵢ)² | 절대적 거리 측정 | 이미지 임베딩, 클러스터링 |
| **Dot Product** | d = Σ(aᵢ·bᵢ) | 크기+방향 모두 반영 | 정규화된 임베딩 |

#### HNSW(Hierarchical navigable small world)알고리즘 직관적 이해

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      상위 레이어에서 대략적 위치 → 하위로 내려가며 정밀 탐색
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/hierarchical navigable small world (HNSW).jpg" width="400" />
    </td>
  </tr>
</table>


* **계층적 그래프** 구조로, 상위 레이어에서 빠르게 후보를 좁히고 하위로 내려가며 정확한 이웃을 찾음
* 검색 시간: **O(log n)** — 10억 개 벡터에서도 밀리초 단위 응답 가능

---

### 4-2. 키워드 검색 (Keyword Search)

정확한 **단어/구문 일치**를 기반으로 문서를 찾는 전통적 검색 방법입니다.

#### 핵심 알고리즘: BM25 (Best Matching 25)

| 항목 | 설명 |
|---|---|
| **기반** | TF-IDF의 개선 버전 |
| **TF (Term Frequency)** | 문서 내 검색어 등장 빈도 (포화 함수 적용) |
| **IDF (Inverse Document Frequency)** | 희귀한 단어일수록 높은 가중치 |
| **문서 길이 정규화** | 긴 문서가 불리하지 않도록 보정 |

#### BM25 점수 계산 직관적 예시

```
쿼리: "레이크하우스 아키텍처"

문서 A: "데이터브릭스 레이크하우스 아키텍처 설계"  → TF(레이크하우스)=1, TF(아키텍처)=1 → ★★ 높은 점수
문서 B: "레이크하우스 소개"                    → TF(레이크하우스)=1, TF(아키텍처)=0 → ★☆ 중간 점수
문서 C: "파이썬 프로그래밍 입문"                  → TF(레이크하우스)=0, TF(아키텍처)=0 → ☆☆ 낮은 점수
```

---

### 4-3. 하이브리드 검색이 필요한 이유

| 시나리오 | Similarity Only | Keyword Only | **Hybrid** |
|---|---|---|---|
| "데이터브릭스란 뭐야?" | ✅ 의미적 유사 문서 발견 | ⚠️ 정확한 단어 없으면 누락 | ✅ 최적 |
| "error code E-1234" | ⚠️ 에러코드 의미 파악 불가 | ✅ 정확한 코드 매칭 | ✅ 최적 |
| "Lakehouse 성능 최적화" | ✅ 관련 문서 발견 | ✅ 키워드 매칭 | ✅✅ 보완적 |

> **핵심**: 유사도 검색은 **의미적 이해**에 강하고, 키워드 검색은 **정확한 용어 매칭**에 강합니다.  
> 하이브리드 검색은 두 가지를 **Score Fusion(가중 합산)**으로 결합하여 양쪽의 단점을 보완합니다.

---

## 5. 다음 실습 안내

| 노트북 | 내용 |
|---|---|
| **02-1. create_vector_search_index** | Vector Search 인덱스 생성 실습 |
| **02-2. use_vector_search_index** | 인덱스 쿼리 및 활용 실습 |
| **03. databricks_langchain** | LangChain 연동 실습 |
| **04. langgraph_vector_search_index** | LangGraph Agent에서 Vector Search 활용 |